# Notebook 26 — Volume, order-flow proxies and market breadth

**Research question:** Can ticker volume and contemporaneous market participation improve our seven-quantile classifier and class-1 entry quality? 3 chronological folds, 4 feature families, 6 models = **72 fits**. No historical holdout access or prospective ledger writes.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise RuntimeError('Open from repository root or notebooks directory')
if str(ROOT/'src') not in sys.path:sys.path.insert(0,str(ROOT/'src'))
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.volume_regime_research import run_volume_regime_research,CONFIGS
print('Project:',ROOT)

Project: /home/oem/Documents/github/stock_ml_dynamic


## 1. Research protocol

Fixed target `q=7`, mapping `[2,2,0,0,1,1,1]`. All transforms refit chronologically. We test balanced and unweighted controls explicitly; the final historical holdout is not evaluated.

In [2]:
N_FOLDS=3
MAX_TRAIN_ROWS=60000
TREE_TRAIN_ROWS=40000
THRESHOLDS=(.45,.50,.55,.60)
COST_BPS=10
OUTPUT=ROOT/'artifacts'/'volume_regime'/'1d'
print('Model fits:',N_FOLDS*4*len(CONFIGS),'threshold diagnostics:',N_FOLDS*4*len(CONFIGS)*len(THRESHOLDS))

Model fits: 72 threshold diagnostics: 288


## 2. Load frozen master data

Load the master DataFrame, **not** the full four-item tuple. The research module internally reserves the historical holdout.

In [3]:
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
paths=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
source=next((x for x in paths if all((x/y).exists() for y in required)),None)
if source is None: raise FileNotFoundError('Master table artifacts not found; run Notebook 03 first')
master,latest,bundle,manifest=load_master(source)
print('Master rows:',len(master),'symbols:',master.symbol.nunique(),'last date:',master.bar_timestamp.max())
print('Volume missing:',round(master.volume.isna().mean(),4))

Master rows: 431271 symbols: 220 last date: 2026-11-09 00:00:00+00:00
Volume missing: 0.0


## 3. Run extensive three-fold experiment

This may take a long time. All 72 fits are intentional. We do not select a threshold from outer-fold results and then claim its performance is unbiased.

In [4]:
results,leaderboard,diagnostics,cutoff=run_volume_regime_research(master,n_folds=N_FOLDS,
    max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,thresholds=THRESHOLDS,
    cost_bps=COST_BPS,output_dir=OUTPUT)
print('Historical development cutoff:',cutoff)
print('Rows:',len(results))
display(diagnostics)

Historical development cutoff: 2025-04-07 00:00:00+00:00
Rows: 288


,fold,n_train,n_test,test_start,test_end,volume_missing_rate,up_base,median_up_return,fraction_up_return_above_cost
0,1,173557,51635,2022-05-25 00:00:00+00:00,2023-05-05 00:00:00+00:00,0.0,0.421788,0.013880,1.0
1,2,225409,51839,2023-05-09 00:00:00+00:00,2024-04-19 00:00:00+00:00,0.0,0.411698,0.010839,1.0
2,3,277464,52247,2024-04-23 00:00:00+00:00,2025-04-04 00:00:00+00:00,0.0,0.425230,0.011468,1.0


## 4. Paired feature-family and model leaderboard

Accuracy is compared with the correct ~42% majority baseline. Entry precision is compared with the actual Up prevalence.

In [5]:
display(leaderboard[['model','feature_set','mean_accuracy','mean_baseline','accuracy_lift','mean_auc','mean_entry_precision','mean_entry_coverage','mean_net_return','mean_fit_seconds']].head(24))
print('No model promoted: these are exploratory development folds.')

,model,feature_set,mean_accuracy,mean_baseline,accuracy_lift,mean_auc,mean_entry_precision,mean_entry_coverage,mean_net_return,mean_fit_seconds
20,random_forest_unweighted,price_only,0.423103,0.419572,0.003531,0.558730,0.441877,0.051992,-0.001531,2.619938
22,random_forest_unweighted,price_plus_volume,0.422511,0.419572,0.002939,0.559687,0.449883,0.059407,-0.001142,3.589185
21,random_forest_unweighted,price_plus_flow,0.421099,0.419572,0.001527,0.559152,0.455804,0.062589,-0.000632,3.108730
23,random_forest_unweighted,volume_only,0.416687,0.419572,-0.002885,0.534722,0.448758,0.056504,-0.000694,2.308450
12,logistic_unweighted,price_only,0.409630,0.419572,-0.009942,0.549657,0.446831,0.140205,-0.000461,6.947430
14,logistic_unweighted,price_plus_volume,0.408281,0.419572,-0.011291,0.549762,0.443570,0.163633,-0.000397,17.024374
13,logistic_unweighted,price_plus_flow,0.407324,0.419572,-0.012247,0.548308,0.443320,0.163961,-0.000453,13.832468
15,logistic_unweighted,volume_only,0.399822,0.419572,-0.019750,0.529666,0.437636,0.156432,-0.000725,3.747286
6,hist_boost_balanced,price_plus_volume,0.383836,0.419572,-0.035736,0.556902,0.461213,0.059250,-0.000206,9.953718
18,random_forest_balanced,price_plus_volume,0.378726,0.419572,-0.040846,0.560896,0.502795,0.003224,-0.003230,3.529377


No model promoted: these are exploratory development folds.


## 5. Which features add signal?

Compare paired fold performance with the same model, not different model configurations. Report accuracy *and* Up-entry behavior.

In [6]:
ref=results[results.entry_threshold.eq(.50)]
pivot=ref.pivot_table(index=['fold','model'],columns='feature_set',values='accuracy')
pivot['volume_minus_price']=pivot['price_plus_volume']-pivot['price_only']
pivot['flow_minus_price']=pivot['price_plus_flow']-pivot['price_only']
display(pivot.sort_values('volume_minus_price',ascending=False))
print('Average paired volume lift:',pivot.volume_minus_price.mean())

feature_set                    price_only  price_plus_flow  price_plus_volume  \
fold model                                                                      
1    hist_boost_balanced         0.374959         0.371027           0.394326   
     random_forest_balanced      0.370795         0.374417           0.374455   
3    hist_boost_balanced         0.377572         0.376232           0.380960   
2    extra_trees_balanced        0.376088         0.376917           0.378962   
3    extra_trees_balanced        0.368959         0.370662           0.371715   
2    logistic_balanced           0.380177         0.380505           0.382550   
3    random_forest_balanced      0.375505         0.374797           0.377668   
1    logistic_unweighted         0.396204         0.394364           0.397463   
2    random_forest_balanced      0.383360         0.383206           0.384054   
     random_forest_unweighted    0.425259         0.422790           0.425375   
3    random_forest_unweighted    0.427202         0.425441           0.426800   
2    logistic_unweighted         0.415093         0.414398           0.414514   
     hist_boost_balanced         0.377148         0.386331           0.376223   
1    extra_trees_balanced        0.369013         0.368394           0.367987   
     random_forest_unweighted    0.416849         0.415067           0.415358   
3    logistic_balanced           0.377400         0.374644           0.374586   
     logistic_unweighted         0.417593         0.413210           0.412866   
1    logistic_balanced           0.376334         0.369614           0.368510   

feature_set                    volume_only  volume_minus_price  \
fold model                                                       
1    hist_boost_balanced          0.376101            0.019367   
     random_forest_balanced       0.358168            0.003660   
3    hist_boost_balanced          0.372462            0.003388   
2    extra_trees_balanced         0.355929            0.002874   
3    extra_trees_balanced         0.372347            0.002756   
2    logistic_balanced            0.361562            0.002373   
3    random_forest_balanced       0.367677            0.002163   
1    logistic_unweighted          0.386927            0.001259   
2    random_forest_balanced       0.365439            0.000694   
     random_forest_unweighted     0.413395            0.000116   
3    random_forest_unweighted     0.424426           -0.000402   
2    logistic_unweighted          0.403750           -0.000579   
     hist_boost_balanced          0.363047           -0.000926   
1    extra_trees_balanced         0.359911           -0.001026   
     random_forest_unweighted     0.412240           -0.001491   
3    logistic_balanced            0.361188           -0.002814   
     logistic_unweighted          0.408789           -0.004728   
1    logistic_balanced            0.353714           -0.007824   

feature_set                    flow_minus_price  
fold model                                       
1    hist_boost_balanced              -0.003931  
     random_forest_balanced            0.003622  
3    hist_boost_balanced              -0.001340  
2    extra_trees_balanced              0.000829  
3    extra_trees_balanced              0.001703  
2    logistic_balanced                 0.000328  
3    random_forest_balanced           -0.000708  
1    logistic_unweighted              -0.001840  
2    random_forest_balanced           -0.000154  
     random_forest_unweighted         -0.002469  
3    random_forest_unweighted         -0.001761  
2    logistic_unweighted              -0.000694  
     hist_boost_balanced               0.009182  
1    extra_trees_balanced             -0.000620  
     random_forest_unweighted         -0.001782  
3    logistic_balanced                -0.002756  
     logistic_unweighted              -0.004383  
1    logistic_balanced                -0.006720

Average paired volume lift: 0.001047791356151764


## 6. Entry threshold diagnostics — descriptive only

Higher precision with vanishing coverage is not a robust entry strategy. Thresholds are **not** selected from these evaluation folds.

In [7]:
display(results.groupby(['model','feature_set','entry_threshold'],as_index=False).agg(
    precision=('entry_precision','mean'),coverage=('entry_coverage','mean'),
    net_return=('mean_entry_net_return','mean'),entries=('entry_count','sum')).sort_values(['precision','coverage'],ascending=False).head(35))

,model,feature_set,entry_threshold,precision,coverage,net_return,entries
2,extra_trees_balanced,price_only,0.55,1.000000,0.000019,0.025266,3
6,extra_trees_balanced,price_plus_flow,0.55,1.000000,0.000013,0.049329,2
66,random_forest_balanced,price_only,0.55,0.671296,0.000334,0.002097,52
9,extra_trees_balanced,price_plus_volume,0.50,0.657350,0.000354,-0.001126,55
5,extra_trees_balanced,price_plus_flow,0.50,0.541219,0.000735,-0.005266,114
95,random_forest_unweighted,volume_only,0.60,0.536805,0.003233,0.004378,501
67,random_forest_balanced,price_only,0.60,0.527778,0.000083,-0.001499,13
65,random_forest_balanced,price_only,0.50,0.523068,0.002079,-0.001040,323
69,random_forest_balanced,price_plus_flow,0.50,0.522436,0.005534,-0.001402,858
14,extra_trees_balanced,volume_only,0.55,0.516576,0.000683,0.001152,106


## 7. Market conditions and economic target diagnostics

Inspect whether Up-label returns clear costs; a Q5 Up outcome can be unprofitable. A new economic target must be specified and evaluated in a future experiment, not selected from these test results.

In [8]:
display(diagnostics[['fold','up_base','median_up_return','fraction_up_return_above_cost','volume_missing_rate']])
print('Data artifacts:',OUTPUT)

,fold,up_base,median_up_return,fraction_up_return_above_cost,volume_missing_rate
0,1,0.421788,0.013880,1.0,0.0
1,2,0.411698,0.010839,1.0,0.0
2,3,0.425230,0.011468,1.0,0.0


Data artifacts: /home/oem/Documents/github/stock_ml_dynamic/artifacts/volume_regime/1d


## 8. Decision rubric

**Advance to a new prospective shadow experiment only if** improvements are positive across all three folds, class-1 precision beats Up prevalence with useful coverage, and net return remains positive under reasonable costs. Otherwise shift to point-in-time sector membership, fundamentals, and news sentiment. Never treat this exploratory leaderboard as proof of trading profitability.